# 나이브 베이즈 - 메일 스팸 여부 판단하기
- 조건부 확률 기반의 분류모델
- p(A|B) B라는 사건이 발생했을 때, A라는 사건이 발생할 확률을 구하기 위해 우변의 변수를 활용하는 것.

In [168]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns

In [169]:
data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/data/spam.csv")

In [170]:
data.head()

,target,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [171]:
data["target"].unique()

array(['ham', 'spam'], dtype=object)

In [172]:
import string

In [173]:
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [174]:
sample_string = data['text'].loc[0]
sample_string

'Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...'

In [175]:
new_string=[]
for i in sample_string:
  if i not in string.punctuation:
    new_string.append(i)
new_string

['G',
 'o',
 ' ',
 'u',
 'n',
 't',
 'i',
 'l',
 ' ',
 'j',
 'u',
 'r',
 'o',
 'n',
 'g',
 ' ',
 'p',
 'o',
 'i',
 'n',
 't',
 ' ',
 'c',
 'r',
 'a',
 'z',
 'y',
 ' ',
 'A',
 'v',
 'a',
 'i',
 'l',
 'a',
 'b',
 'l',
 'e',
 ' ',
 'o',
 'n',
 'l',
 'y',
 ' ',
 'i',
 'n',
 ' ',
 'b',
 'u',
 'g',
 'i',
 's',
 ' ',
 'n',
 ' ',
 'g',
 'r',
 'e',
 'a',
 't',
 ' ',
 'w',
 'o',
 'r',
 'l',
 'd',
 ' ',
 'l',
 'a',
 ' ',
 'e',
 ' ',
 'b',
 'u',
 'f',
 'f',
 'e',
 't',
 ' ',
 'C',
 'i',
 'n',
 'e',
 ' ',
 't',
 'h',
 'e',
 'r',
 'e',
 ' ',
 'g',
 'o',
 't',
 ' ',
 'a',
 'm',
 'o',
 'r',
 'e',
 ' ',
 'w',
 'a',
 't']

In [176]:
new_string = ''.join(new_string)
new_string

'Go until jurong point crazy Available only in bugis n great world la e buffet Cine there got amore wat'

In [177]:
# 특수기호 처리
def remove_punc(x):
  new_string=[]
  for i in x:
    if i not in string.punctuation:
      new_string.append(i)
  new_string = ''.join(new_string)
  return new_string

In [178]:
data['text']=data['text'].apply(remove_punc)
data['text']

,text
0,Go until jurong point crazy Available only in ...
1,Ok lar Joking wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...
3,U dun say so early hor U c already then say
4,Nah I dont think he goes to usf he lives aroun...
...,...
5569,This is the 2nd time we have tried 2 contact u...
5570,Will ü b going to esplanade fr home
5571,Pity was in mood for that Soany other suggest...
5572,The guy did some bitching but I acted like id ...


In [179]:
# 불용어 제거하기
# 저장된 문자열을 하나의 단어 단위로 리스트 변환
# 불용어 아니면 소문자로 지정
# 문자를 문자열로 합침

import nltk

from nltk.corpus import stopwords

In [180]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [181]:
# stopwords에서 제공되는 리스트를 의미
# 24개국어의 불용어 제공
# 한국어 불포함
stopwords.words('english')

['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [182]:
# 특수기호 처리
def remove_word(x):
  nopewords=[]
  for i in x.split():
    if i.lower() not in stopwords.words('english'):
      nopewords.append(i)
  nopewords = ' '.join(nopewords)
  return nopewords

In [183]:
data['text']=data['text'].apply(remove_word)
data['text']

,text
0,Go jurong point crazy Available bugis n great ...
1,Ok lar Joking wif u oni
2,Free entry 2 wkly comp win FA Cup final tkts 2...
3,U dun say early hor U c already say
4,Nah dont think goes usf lives around though
...,...
5569,2nd time tried 2 contact u U £750 Pound prize ...
5570,ü b going esplanade fr home
5571,Pity mood Soany suggestions
5572,guy bitching acted like id interested buying s...


In [184]:
# 텍스트를 숫자로 변환
data['target'] = data['target'].map({'spam':1,'ham':0})
data['target']

,target
0,0
1,0
2,1
3,0
4,0
...,...
5569,1
5570,0
5571,0
5572,0


In [185]:
from sklearn.feature_extraction.text import CountVectorizer

In [186]:
x = data['text']
y = data['target']

In [187]:
cv = CountVectorizer()
cv.fit(x)
cv.vocabulary_

{'go': 3791,
 'jurong': 4687,
 'point': 6433,
 'crazy': 2497,
 'available': 1414,
 'bugis': 1881,
 'great': 3888,
 'world': 9184,
 'la': 4847,
 'buffet': 1879,
 'cine': 2214,
 'got': 3848,
 'amore': 1181,
 'wat': 8947,
 'ok': 5995,
 'lar': 4886,
 'joking': 4655,
 'wif': 9079,
 'oni': 6027,
 'free': 3577,
 'entry': 3160,
 'wkly': 9136,
 'comp': 2330,
 'win': 9093,
 'fa': 3296,
 'cup': 2553,
 'final': 3421,
 'tkts': 8380,
 '21st': 454,
 'may': 5335,
 '2005': 441,
 'text': 8217,
 '87121': 875,
 'receive': 6833,
 'questionstd': 6724,
 'txt': 8592,
 'ratetcs': 6776,
 'apply': 1267,
 '08452810075over18s': 71,
 'dun': 3011,
 'say': 7192,
 'early': 3031,
 'hor': 4222,
 'already': 1154,
 'nah': 5682,
 'dont': 2918,
 'think': 8291,
 'goes': 3805,
 'usf': 8741,
 'lives': 5050,
 'around': 1318,
 'though': 8310,
 'freemsg': 3585,
 'hey': 4118,
 'darling': 2617,
 'weeks': 9002,
 'word': 9170,
 'back': 1464,
 'id': 4343,
 'like': 5000,
 'fun': 3652,
 'still': 7860,
 'tb': 8147,
 'xxx': 9309,
 'std': 

In [188]:
x = cv.transform(x)
print(x)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 47493 stored elements and shape (5574, 9437)>
  Coords	Values
  (0, 1181)	1
  (0, 1414)	1
  (0, 1879)	1
  (0, 1881)	1
  (0, 2214)	1
  (0, 2497)	1
  (0, 3791)	1
  (0, 3848)	1
  (0, 3888)	1
  (0, 4687)	1
  (0, 4847)	1
  (0, 6433)	1
  (0, 8947)	1
  (0, 9184)	1
  (1, 4655)	1
  (1, 4886)	1
  (1, 5995)	1
  (1, 6027)	1
  (1, 9079)	1
  (2, 71)	1
  (2, 441)	1
  (2, 454)	1
  (2, 875)	1
  (2, 1267)	1
  (2, 2330)	1
  :	:
  (5570, 3188)	1
  (5570, 3564)	1
  (5570, 3810)	1
  (5570, 4188)	1
  (5571, 5566)	1
  (5571, 6359)	1
  (5571, 7611)	1
  (5571, 7986)	1
  (5572, 999)	1
  (5572, 1665)	1
  (5572, 1916)	1
  (5572, 3103)	1
  (5572, 3577)	1
  (5572, 3701)	1
  (5572, 3950)	1
  (5572, 4343)	1
  (5572, 4480)	1
  (5572, 5000)	1
  (5572, 5777)	1
  (5572, 7636)	1
  (5572, 8731)	1
  (5572, 8997)	1
  (5573, 5688)	1
  (5573, 7052)	1
  (5573, 8538)	1


In [189]:
from sklearn.feature_extraction.text import CountVectorizer

In [190]:
data.loc[0]['text']

'Go jurong point crazy Available bugis n great world la e buffet Cine got amore wat'

In [191]:
print(cv.vocabulary_['go'])
print(cv.vocabulary_['jurong'])

3791
4687


In [196]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score,confusion_matrix
X_train,X_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=100)

In [197]:
model = MultinomialNB()
model.fit(X_train,y_train)
pred = model.predict(X_test)
accuracy_score(y_test,pred)

0.9856502242152466